In [12]:
import pandas as pd

# 1. 파일 경로
file1_path = "식당대12중53소132상세메뉴380분류.csv"
file2_path = "키워드_검색수_평균_결과_순서유지.xlsx"
file3_path = "병합된_상세메뉴_월별추세.xlsx"

# 2. 첫 번째 파일: 상세메뉴 분리 (순서 유지)
df_menu = pd.read_csv(file1_path)
menu_list = []
for line in df_menu["상세메뉴"].dropna():
    items = [item.strip() for item in line.split(",")]
    menu_list.extend(items)
df_base = pd.DataFrame({"상세메뉴": menu_list})

# 3. 두 번째 파일: 평균 검색수
df_avg = pd.read_excel(file2_path)

# 4. 세 번째 파일: 월별 추세 transpose 및 월평균
df_monthly = pd.read_excel(file3_path)
df_transposed = df_monthly.set_index("월").transpose()
df_transposed.index.name = "상세메뉴"
df_transposed.reset_index(inplace=True)
month_columns = [col for col in df_transposed.columns if col != "상세메뉴"]
df_transposed[month_columns] = df_transposed[month_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
df_transposed["월평균"] = df_transposed[month_columns].mean(axis=1)

# 5. 포함 기준 매칭
def find_match(name, candidates):
    for c in candidates:
        if name in c:
            return c
    return None

df_base["상세메뉴_검색수파일"] = df_base["상세메뉴"].apply(lambda x: find_match(x, df_avg["상세메뉴"]))
df_base = pd.merge(df_base, df_avg, left_on="상세메뉴_검색수파일", right_on="상세메뉴", how="left", suffixes=("", "_검색수"))

df_base["상세메뉴_추세파일"] = df_base["상세메뉴"].apply(lambda x: find_match(x, df_transposed["상세메뉴"]))
df_base = pd.merge(df_base, df_transposed, left_on="상세메뉴_추세파일", right_on="상세메뉴", how="left", suffixes=("", "_추세"))

# 6. 보정비율 계산
def compute_scaling(row):
    if pd.isna(row["월평균"]) or row["월평균"] < 1e-6:
        return row["총합"] * 0.5 if pd.notna(row["총합"]) else None
    return row["총합"] / row["월평균"] if pd.notna(row["총합"]) else None

df_base["보정비율"] = df_base.apply(compute_scaling, axis=1)

# 7. 보정 추세값 계산
for col in month_columns:
    df_base[f"{col}_보정"] = df_base[col] * df_base["보정비율"]

# 8. 월평균이 0인 항목의 추세값 일괄 대체
mask = df_base["월평균"].isna() | (df_base["월평균"] < 1e-6)
for col in month_columns:
    df_base.loc[mask, f"{col}_보정"] = df_base.loc[mask, "총합"] * 0.5

# 9. 병합 상태 로그
df_base["검색수_매칭여부"] = df_base["상세메뉴_검색수파일"].notna()
df_base["추세_매칭여부"] = df_base["상세메뉴_추세파일"].notna()
df_base["보정_성공"] = df_base["보정비율"].notna()

# 10. 월별 총합 기준 비중 계산
for col in month_columns:
    보정값_컬럼 = f"{col}_보정"
    비중_컬럼 = f"{col}_비중"
    total = df_base[보정값_컬럼].sum()
    df_base[비중_컬럼] = df_base[보정값_컬럼] / total if total > 0 else 0

# 11. 최종 결과 정리
columns_to_export = (
    ["상세메뉴", "상세메뉴_검색수파일", "상세메뉴_추세파일", "총합", "월평균", "보정비율",
     "검색수_매칭여부", "추세_매칭여부", "보정_성공"] +
    [f"{col}_보정" for col in month_columns] +
    [f"{col}_비중" for col in month_columns]
)

df_result = df_base[columns_to_export]

# 12. 저장
df_result.to_excel("상세메뉴_보정결과_비중포함_순서유지.xlsx", index=False)